# 13 · Layer 4 — Per-Claim Agentic Retrieval
Hard claim filter → scoped vector entry → 1–2 hop graph expansion → grounded synthesis with span citations. Includes the scope-isolation proof: the agent is structurally incapable of reading another claim's data.

In [ ]:
# --- bootstrap: make the src package importable from any working dir ---
import sys
from pathlib import Path
p = Path.cwd().resolve()
while not (p / 'config' / '00_config.py').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
print('project root:', p)


In [ ]:
from src.repository import Repository
from src.agent import ClaimScopedAgent, test_scope_isolation
repo = Repository()
agent = ClaimScopedAgent(repo)
res = agent.answer('CLM0005', 'who represents the claimant and which providers treated them?')
print('SCOPE:', res['scope'])
print('\nparties:', [(e['name'], e['class']) for e in res['entities']][:8])
print('triples retrieved:', len(res['triples']))
print('\nANSWER:\n', res['answer'])
print('\ncitations:', res['citations'][:5])


In [ ]:
# every retrieval step, shown
chunks = agent.retrieve_chunks('CLM0005', 'attorney representation and treatment')
print('step 2 — scoped vector entry:')
for c in chunks:
    print(f"   {c['chunk_id']}  claim={c['claim_id']}  score={c['score']}")
eids = agent.entities_in_chunks('CLM0005', chunks)
print('\nstep 3 — entities in those chunks:', len(eids))
for t in agent.expand('CLM0005', eids)[:8]:
    print(f"   {t['subject'][:24]:26s} --{t['predicate']:20s}--> {t['object'][:22]}")


In [ ]:
# SCOPE ISOLATION PROOF
iso = test_scope_isolation(agent, 'CLM0005', 'CLM0006')
for k, v in iso.items():
    print(f'  {k:34s} {v}')
assert iso['isolation_holds'], 'SCOPE ISOLATION FAILED'
print('\nscope isolation holds.')


In [ ]:
# escalated cross-claim view (fraud network) — separately authorized
ents = [e['entity_id'] for e in res['entities']]
links = agent.cross_claim_network(ents, authorized=True)
print('cross-claim links for these entities:', len(links))
for l in links[:5]:
    print(f"   {l['subject'][:22]:24s} --{l['predicate']:24s}--> {l['object'][:22]:24s}")
    print(f"      subject claims: {l['claims_of_subject'][:4]}  object claims: {l['claims_of_object'][:4]}")
repo.close()
